# Suite LOG — The logical layout

`Logical` defines a diagram's logical layout: diagrams, shapes, points, vectors, segments and adjacency specs, each a reference object owned through `Logical.Owns`, each playing its part through a relation of its own, with no coordinates; and resolves the compass of a point or a shape: mbse-basis [CAP-237: logical layout schemas](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-237.md), by the rules of [the diagrams proposal](https://github.com/pitaman71/mbse-basis/blob/main/proposals/DIAGRAMS.md).

In [ ]:
import json

from mbse.Schemas.Framework import JSON, Builders, Paths, Validators
from mbse.Diagrams.Framework import Logical
from support import text

store = Builders.Store()
store.transaction(Logical.install)
L = store.Logical
directory = store.singleton("Logical.Diagrams")
problems = Validators.Validate(store).Reachable


def owned(owner, name):
    """The entry by which `owner` owns what it builds, as `name`."""
    return lambda a: a.insert(lambda e: e.name(name).owner(owner))


def diagram(name):
    return L.OfDiagram(lambda d: d.owner(owned(directory, name)).directory(lambda a: a.insert(lambda e: e.directory(directory))))


def shape(container, name):
    """A shape owned by `container`, a diagram or a shape, and in it."""
    role = (lambda s: s.diagram(lambda a: a.insert(lambda e: e.diagram(container)))) \
        if container.as_schema().name == "Logical.OfDiagram" else \
        (lambda s: s.container(lambda a: a.insert(lambda e: e.container(container))))
    return L.OfShape(lambda s: role(s.owner(owned(container, name))))


def point(of, name, role):
    """A point owned by the shape `of`, as its origin, a border or an extent point."""
    link = {"origin": "origin_of", "border": "border_of", "extent": "extent_of"}[role]
    return L.OfPoint(lambda p: getattr(p.owner(owned(of, name)), link)(lambda a: a.insert(lambda e: e.shape(of))))


def vector(of, name, start, end, degrees=None, length=1.0):
    def spec(v):
        v = v.owner(owned(of, name)).start(lambda a: a.insert(lambda e: e.point(start)))
        v = v.end(lambda a: a.insert(lambda e: e.point(end))).length(length)
        return v.rotation(lambda r: r.degrees(degrees)) if degrees is not None else v
    return L.OfVector(spec)


def raises(message, block):
    try:
        with store.transaction():
            block()
    except ValueError as error:
        assert str(error) == message, str(error)
    else:
        raise AssertionError(f"expected ValueError: {message}")


path = lambda x: JSON.ToJSON(store)(store.Paths.OfRoot.as_schema(), Paths.of(x))
names = lambda owner: [e.name.get() for e in owner.owns]

<a id="LOG-01"></a>
## LOG-01 · The install defines the logical layout's schemas, each valid, and run again writes nothing

In [ ]:
NAMES = ['Logical.Owns', 'Logical.OfRotation', 'Logical.DiagramDirectory', 'Logical.InDiagrams', 'Logical.OfDiagram', 'Logical.InDiagram', 'Logical.OfShape', 'Logical.Containment', 'Logical.OfPoint', 'Logical.Origin', 'Logical.Borders', 'Logical.Extents', 'Logical.OfVector', 'Logical.VectorStart', 'Logical.VectorEnd', 'Logical.OfSegment', 'Logical.SegmentEnds', 'Logical.OfAdjacency', 'Logical.AdjacencyFrom', 'Logical.AdjacencyTo']
schemas = [store.registered(name) for name in NAMES]
assert [schema.validate() for schema in schemas] == [[]] * len(NAMES)
store.transaction(Logical.install)
assert [store.registered(name) for name in NAMES] == schemas and directory is store.Instances.Logical.Diagrams

<a id="LOG-02"></a>
## LOG-02 · Everything is owned through Logical.Owns, by a name unique among its owner's, its path read back through it

In [ ]:
with store.transaction():
    overview = diagram("overview")
    cpu = shape(overview, "cpu")
    alu = shape(cpu, "alu")
    o, ne, e = point(cpu, "origin", "origin"), point(cpu, "ne", "border"), point(cpu, "e", "extent")
    vector(cpu, "to ne", o, ne, 45.0, 2.0)
    vector(cpu, "to e", o, e, 90.0)
    point(alu, "origin", "origin")
assert text([names(directory), names(overview), names(cpu), names(alu)]) == '[["overview"],["cpu"],["alu","origin","ne","e","to ne","to e"],["origin"]]'
assert text([Logical.owner_of(x) is owner for x, owner in ((overview, directory), (cpu, overview), (alu, cpu), (o, cpu))]) == '[true,true,true,true]'
assert text(path(alu.owns({"name": "origin"}))) == '"{\\"singleton\\": \\"Logical.Diagrams\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"owns\\", \\"o\\": \\"owner\\", \\"properties\\": {\\"name\\": \\"overview\\"}, \\"next\\": {\\"entry\\": {\\"i\\": \\"owns\\", \\"o\\": \\"owner\\", \\"properties\\": {\\"name\\": \\"cpu\\"}, \\"next\\": {\\"entry\\": {\\"i\\": \\"owns\\", \\"o\\": \\"owner\\", \\"properties\\": {\\"name\\": \\"alu\\"}, \\"next\\": {\\"entry\\": {\\"i\\": \\"owns\\", \\"o\\": \\"owner\\", \\"properties\\": {\\"name\\": \\"origin\\"}, \\"next\\": {\\"target\\": {}}}}}}}}}}}"'
assert text([Logical.owner_of(directory), Logical.name_of(directory), Logical.name_of(cpu)]) == '[null,null,"cpu"]'
assert text(problems(L.OfDiagram, overview)) == '[]'

<a id="LOG-03"></a>
## LOG-03 · A shape's roles: in its diagram or its container, one origin, and border and extent points; refused without an owner

In [ ]:
assert text([[x.shape is cpu for x in overview.shapes], [x.child is alu for x in cpu.children], [x.container is cpu for x in alu.container]]) == '[[true],[true],[true]]'
assert text([[x.point is o for x in cpu.origin], [x.point is ne for x in cpu.borders], [x.point is e for x in cpu.extents]]) == '[[true],[true],[true]]'
with store.transaction():
    point(cpu, "second", "origin")
assert text(problems(L.OfShape, cpu)) == '["unique(point) violated: entries agreeing on [\'shape\'] differ on [\'point\']"]'
with store.transaction():
    L.OfShape(cpu, lambda s: s.owns(lambda a: a.remove(lambda e: e.name("second"))))
assert text([problems(L.OfShape, cpu), names(cpu)]) == '[[],["alu","origin","ne","e","to ne","to e"]]'
raises("a 'Logical.OfPoint' needs an owner through 'owner': give it one, since no singleton can be",
       lambda: L.OfPoint(lambda p: p))

<a id="LOG-04"></a>
## LOG-04 · A vector runs from one start to one end, by a rotation and a length

In [ ]:
v = cpu.owns({"name": "to ne"})
assert text([Logical.ends(v)[0] is o, Logical.ends(v)[1] is ne, v.rotation.degrees.get(), v.length.get()]) == '[true,true,45.0,2.0]'
assert text([[x.vector is v for x in ne.incoming], len(list(o.outgoing))]) == '[[true],2]'
with store.transaction():
    L.OfVector(v, lambda x: x.end(lambda a: a.insert(lambda e: e.point(o))))
assert text(problems(L.OfVector, v)) == '["unique(point) violated: entries agreeing on [\'vector\'] differ on [\'point\']"]'
with store.transaction():
    L.OfVector(v, lambda x: x.end(lambda a: a.remove(lambda e: e.point(o))))
assert text(problems(L.OfVector, v)) == '[]'

<a id="LOG-05"></a>
## LOG-05 · A segment joins two points, with no length or rotation; an adjacency spec runs from one shape to another

In [ ]:
with store.transaction():
    seg = L.OfSegment(lambda s: s.owner(owned(cpu, "edge")).ends(lambda a: a.insert(lambda x: x.point(ne)).insert(
        lambda x: x.point(e))))
    fpu = shape(cpu, "fpu")
    spec = L.OfAdjacency(lambda s: s.owner(owned(alu, "to fpu 90")).from_shape(lambda a: a.insert(lambda x: x.shape(alu)))
                         .to_shape(lambda a: a.insert(lambda x: x.shape(fpu))).rotation(lambda r: r.degrees(90.0)))
assert text([[x.point is p for x, p in zip(seg.ends, (ne, e))], [x.segment is seg for x in e.segments]]) == '[[true,true],[true]]'
assert text([[x.adjacency is spec for x in alu.adjacent_to], [x.adjacency is spec for x in fpu.adjacent_from], spec.rotation.degrees.get(), names(alu)]) == '[[true],[true],90.0,["origin","to fpu 90"]]'
assert text(problems(L.OfDiagram, overview)) == '[]'

<a id="LOG-06"></a>
## LOG-06 · A compass: a shape's and its origin's north, a border or extent point's its vector's rotation

In [ ]:
with store.transaction():
    sw = point(cpu, "sw", "extent")
    vector(cpu, "to sw", o, sw, 585.5)
    w = point(cpu, "w", "border")
    vector(cpu, "to w", o, w, -90.0)
assert text([Logical.compass(x) for x in (cpu, o, alu, ne, e, sw, w)]) == '[0.0,0.0,0.0,45.0,90.0,225.5,270.0]'

<a id="LOG-07"></a>
## LOG-07 · A compass the rules leave undefined is refused

In [ ]:
with store.transaction():
    bare = shape(cpu, "bare")
    x = point(bare, "x", "border")
    q = point(fpu, "q", "border")
    point(fpu, "origin", "origin")
    s = point(cpu, "s", "border")
    vector(cpu, "to s", o, s)
    t = point(cpu, "t", "border")
    vector(cpu, "to t 1", o, t, 1.0)
    vector(cpu, "to t 2", o, t, 2.0)
raises("a shape without an origin has no compass", lambda: Logical.compass(bare))
raises("a shape without an origin has no compass", lambda: Logical.compass(x))
raises("a point with 0 vectors from its shape's origin, not one, has no compass", lambda: Logical.compass(q))
raises("a point with 2 vectors from its shape's origin, not one, has no compass", lambda: Logical.compass(t))
raises("a point whose vector from its shape's origin has no rotation has no compass", lambda: Logical.compass(s))